In [1]:
import argparse
import importlib
import json
import os
import sys
from typing import Any, Dict

import torch
import torch.nn as nn


In [2]:
def human_int(n: int) -> str:
    return f"{n:,}"


In [3]:
def import_object(qualname: str) -> Any:
    """
    'module.sub:obj' 또는 'module.sub.obj' 형태를 import 해서 반환.
    함수/클래스/모듈 다 가능.
    """
    if ":" in qualname:
        mod, name = qualname.split(":", 1)
    else:
        parts = qualname.split(".")
        mod, name = ".".join(parts[:-1]), parts[-1]
    m = importlib.import_module(mod)
    return getattr(m, name)

In [4]:
def build_model(model_spec: str, model_args: Dict[str, Any]) -> nn.Module:
    obj = import_object(model_spec)
    if callable(obj):
        return obj(**(model_args or {}))
    if isinstance(obj, nn.Module):
        return obj
    raise TypeError(f"model-spec '{model_spec}'는 호출 가능하거나 nn.Module이어야 합니다.")

In [5]:
def print_param_table(model: nn.Module, trainable_only: bool):
    print("\n[Parameter table]")
    print(f"{'name':60} {'shape':>20} {'req_grad':>9} {'#params':>12}")
    print("-" * 110)
    total = 0
    total_trainable = 0
    for name, p in model.named_parameters():
        if trainable_only and not p.requires_grad:
            continue
        cnt = p.numel()
        total += cnt
        if p.requires_grad:
            total_trainable += cnt
        print(f"{name:60} {str(tuple(p.shape)):>20} {str(p.requires_grad):>9} {human_int(cnt):>12}")
    print("-" * 110)
    print(f"Total params:       {human_int(sum(p.numel() for p in model.parameters()))}")
    print(f"Trainable params:   {human_int(sum(p.numel() for p in model.parameters() if p.requires_grad))}")
    print(f"Non-trainable:      {human_int(sum(p.numel() for p in model.parameters() if not p.requires_grad))}")

In [6]:
def print_children_tree(m: nn.Module, prefix: str = ""):
    # 간단한 트리 출력 (이름 포함)
    children = list(m.named_children())
    if not children:
        return
    for i, (name, child) in enumerate(children):
        last = i == len(children) - 1
        branch = "└─ " if last else "├─ "
        print(prefix + branch + f"[{name}] {child.__class__.__name__}")
        print_children_tree(child, prefix + ("   " if last else "│  "))

In [7]:
def try_load_torchscript(path: str, map_location: str):
    try:
        ts = torch.jit.load(path, map_location=map_location)
        return ts
    except Exception:
        return None

In [8]:
def load_checkpoint_any(path: str, map_location: str):
    """
    TorchScript -> nn.Module -> state_dict 단계로 시도
    """
    # 1) TorchScript (.pt/.ts)
    ts = try_load_torchscript(path, map_location)
    if ts is not None:
        return ("torchscript", ts)

    # 2) torch.load (전체 모델 or ckpt dict)
    obj = torch.load(path, map_location=map_location)
    if isinstance(obj, nn.Module):
        return ("module", obj)
    if isinstance(obj, dict):
        return ("state_dict", obj)
    raise TypeError(f"알 수 없는 체크포인트 형식: {type(obj)}")

In [13]:
def load_and_print_model(
    ckpt_path: str,
    model_spec: str = None,
    model_args: dict = None,
    map_location: str = "cpu",
    strict: bool = False,
):
    """ipynb에서 실행용: 모델 로드하고 구조 출력"""

    obj = torch.load(ckpt_path, map_location=map_location)

    if isinstance(obj, nn.Module):
        model = obj
        print("=== nn.Module (saved whole model) ===")
    elif isinstance(obj, dict):
        if model_spec is None:
            raise ValueError("state_dict 체크포인트는 model_spec이 필요합니다")
        model = build_model(model_spec, model_args)
        state = obj.get("state_dict", obj)
        missing, unexpected = model.load_state_dict(state, strict=strict)
        print(f"Missing keys: {missing}")
        print(f"Unexpected keys: {unexpected}")
    else:
        raise TypeError(f"알 수 없는 타입: {type(obj)}")

    print("\n[Model repr]")
    print(model)

    print("\n[Module tree]")
    print(f"{model.__class__.__name__}")
    print_children_tree(model)

    print_param_table(model, trainable_only=False)

In [14]:
load_and_print_model("/mnt/ex_ssd/starcraft-vision/models/all_correct_win4_b8_20250826_044921/model_015.pth", model_spec="model.maskrcnn_builder:get_model_instance_segmentation", model_args={"num_classes":2, "window_size": 4}, strict=False)

Using 36 input channels (window size: 4)
Missing keys: []
Unexpected keys: []

[Model repr]
MaskRCNN(
  (transform): CustomRCNNTransform(
      Normalize(mean=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])
      Resize(min_size=[800], max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(36, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d